# COCO Text Embeddings Cache Generation

This notebook generates pre-encoded text embeddings for COCO 2014 captions using the CLIP text encoder.

**Purpose:** Run this notebook once to generate cached embeddings (~1.2GB total), then upload the output files as a Kaggle dataset for reuse in training.

**Output Files:**
- `train_text_embeddings.pt` (~850 MB)
- `val_text_embeddings.pt` (~410 MB)

These files will be saved to `/kaggle/working/` (on Kaggle) or `./outputs/` (locally).


In [ ]:
import os
import json
from pathlib import Path
from tqdm import tqdm

import torch
from transformers import CLIPProcessor, CLIPModel


## Path Configuration

This notebook auto-detects whether it's running on Kaggle or locally and sets paths accordingly.

**For local execution:** Modify the paths below to point to your COCO dataset location.


In [ ]:
# Auto-detect environment
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    print("Running on Kaggle")
    CAPTIONS_ROOT = Path("/kaggle/input/ms-coco2014/annotations")
    OUTPUT_DIR = Path("/kaggle/working")
else:
    print("Running locally")
    # Modify these paths for local execution
    CAPTIONS_ROOT = Path("./data/annotations")
    OUTPUT_DIR = Path("./outputs")

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set paths to caption files
CAPTIONS_TRAIN_PATH = CAPTIONS_ROOT / "captions_train2014.json"
CAPTIONS_VAL_PATH = CAPTIONS_ROOT / "captions_val2014.json"

print(f"\nCaption files:")
print(f"  Train: {CAPTIONS_TRAIN_PATH}")
print(f"  Val:   {CAPTIONS_VAL_PATH}")
print(f"\nOutput directory: {OUTPUT_DIR}")


## Helper Functions


In [ ]:
def load_coco_captions(json_path: Path):
    """
    Load COCO-style caption JSON (robust to 'root' wrapper).
    
    Some COCO JSON files have a 'root' wrapper - this function handles both formats.
    """
    with json_path.open("r") as f:
        data = json.load(f)

    # Strip 'root' wrapper if present
    if isinstance(data, dict) and "root" in data and isinstance(data["root"], dict):
        data = data["root"]

    # Validate structure
    assert isinstance(data, dict), "Expected top-level JSON object"
    assert "images" in data, "Expected 'images' key in captions file"
    assert "annotations" in data, "Expected 'annotations' key in captions file"

    return data


def encode_captions(
    annotations_path: Path,
    model_name: str = 'openai/clip-vit-base-patch32',
    batch_size: int = 32,
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
) -> dict:
    """
    Encode all captions in COCO annotations using CLIP text encoder.
    
    Args:
        annotations_path: Path to COCO captions JSON file
        model_name: HuggingFace model name for CLIP
        batch_size: Batch size for encoding
        device: Device to run encoding on
        
    Returns:
        dict: {image_id: [list of normalized caption embeddings (512-dim)]}
    """
    print(f"\nLoading CLIP model: {model_name}")
    print(f"Using device: {device}")
    
    # Load CLIP model and processor
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)
    
    # Move model to device and set to eval mode
    model = model.to(device)
    model.eval()
    
    # Freeze all parameters (text encoder will not be trained)
    for param in model.parameters():
        param.requires_grad = False
    
    print("✓ CLIP model loaded and frozen")
    
    # Load annotations
    print(f"\nLoading annotations from {annotations_path}")
    data = load_coco_captions(annotations_path)
    
    # Group captions by image_id
    captions_by_image = {}
    for ann in data['annotations']:
        image_id = ann['image_id']
        caption = ann['caption']
        if image_id not in captions_by_image:
            captions_by_image[image_id] = []
        captions_by_image[image_id].append(caption)
    
    print(f"✓ Found {len(captions_by_image)} images with {len(data['annotations'])} total captions")
    
    # Prepare batched caption list
    print("\nEncoding captions in batches...")
    embeddings_dict = {}
    
    all_image_ids = list(captions_by_image.keys())
    all_captions = []
    caption_to_image_id = []
    
    for image_id in all_image_ids:
        captions = captions_by_image[image_id]
        for caption in captions:
            all_captions.append(caption)
            caption_to_image_id.append(image_id)
    
    # Encode in batches
    with torch.no_grad():
        for i in tqdm(range(0, len(all_captions), batch_size), desc="Encoding batches"):
            batch_captions = all_captions[i:i+batch_size]
            batch_image_ids = caption_to_image_id[i:i+batch_size]
            
            # Tokenize and encode
            inputs = processor(text=batch_captions, return_tensors="pt", padding=True, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            text_outputs = model.get_text_features(**inputs)
            # Normalize embeddings (CLIP uses L2-normalized embeddings)
            text_embeddings = text_outputs / text_outputs.norm(dim=-1, keepdim=True)
            
            # Store embeddings
            for j, image_id in enumerate(batch_image_ids):
                if image_id not in embeddings_dict:
                    embeddings_dict[image_id] = []
                # Move to CPU to save memory
                embeddings_dict[image_id].append(text_embeddings[j].cpu())
    
    print(f"✓ Encoded {len(all_captions)} captions for {len(embeddings_dict)} images")
    return embeddings_dict


def save_embeddings_cache(embeddings_dict: dict, cache_path: Path):
    """
    Save text embeddings to .pt cache file.
    
    Args:
        embeddings_dict: Dictionary mapping image_id to list of caption embeddings
        cache_path: Path to save cache file
    """
    torch.save(embeddings_dict, cache_path)
    file_size_mb = cache_path.stat().st_size / (1024 * 1024)
    print(f"✓ Saved embeddings cache to {cache_path} ({file_size_mb:.1f} MB)")


In [ ]:
print("=" * 60)
print("PROCESSING TRAINING SET")
print("=" * 60)

train_cache_path = OUTPUT_DIR / 'train_text_embeddings.pt'

if train_cache_path.exists():
    print(f"\n⚠ Cache already exists at {train_cache_path}")
    print("Skipping training set. Delete the file to regenerate.")
else:
    train_embeddings = encode_captions(CAPTIONS_TRAIN_PATH, batch_size=64)
    save_embeddings_cache(train_embeddings, train_cache_path)
    
    # Validate embeddings
    sample_emb = next(iter(train_embeddings.values()))[0]
    print(f"\nValidation:")
    print(f"  Embedding shape: {sample_emb.shape}")
    print(f"  Embedding dtype: {sample_emb.dtype}")
    print(f"  L2 norm: {sample_emb.norm().item():.6f} (should be ~1.0)")


## Process Validation Set


In [ ]:
print("\n" + "=" * 60)
print("PROCESSING VALIDATION SET")
print("=" * 60)

val_cache_path = OUTPUT_DIR / 'val_text_embeddings.pt'

if val_cache_path.exists():
    print(f"\n⚠ Cache already exists at {val_cache_path}")
    print("Skipping validation set. Delete the file to regenerate.")
else:
    val_embeddings = encode_captions(CAPTIONS_VAL_PATH, batch_size=64)
    save_embeddings_cache(val_embeddings, val_cache_path)
    
    # Validate embeddings
    sample_emb = next(iter(val_embeddings.values()))[0]
    print(f"\nValidation:")
    print(f"  Embedding shape: {sample_emb.shape}")
    print(f"  Embedding dtype: {sample_emb.dtype}")
    print(f"  L2 norm: {sample_emb.norm().item():.6f} (should be ~1.0)")


## Summary


In [ ]:
print("\n" + "=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)

if train_cache_path.exists():
    train_embeddings = torch.load(train_cache_path, map_location='cpu')
    total_train_captions = sum(len(embs) for embs in train_embeddings.values())
    train_size_mb = train_cache_path.stat().st_size / (1024 * 1024)
    print(f"\nTraining set:")
    print(f"  Images: {len(train_embeddings):,}")
    print(f"  Captions: {total_train_captions:,}")
    print(f"  File size: {train_size_mb:.1f} MB")

if val_cache_path.exists():
    val_embeddings = torch.load(val_cache_path, map_location='cpu')
    total_val_captions = sum(len(embs) for embs in val_embeddings.values())
    val_size_mb = val_cache_path.stat().st_size / (1024 * 1024)
    print(f"\nValidation set:")
    print(f"  Images: {len(val_embeddings):,}")
    print(f"  Captions: {total_val_captions:,}")
    print(f"  File size: {val_size_mb:.1f} MB")

print(f"\nCache files saved to: {OUTPUT_DIR}")
print(f"  - {train_cache_path.name}")
print(f"  - {val_cache_path.name}")

print("\n" + "=" * 60)
print("NEXT STEPS:")
print("=" * 60)
if IS_KAGGLE:
    print("1. Download the cache files from /kaggle/working/")
    print("2. Create a new Kaggle dataset and upload both .pt files")
    print("3. Use this dataset as an input in your training notebook")
else:
    print("1. Upload the cache files to Kaggle as a new dataset")
    print("2. Use this dataset as an input in your training notebook")
